# Data preparation
Publisher downloads, audits, duplicate protection, train/validation/test splits, and validation roles.

In [1]:
from pathlib import Path

# Resolve the project from either its root or a notebook's folder.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [current_directory, *current_directory.parents]
    if (path / "requirements.txt").is_file() and (path / "data_preparation").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Keep the three project folders and requirements.txt together.")
DATA_DIR = PROJECT_ROOT / "data_preparation"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments"
METRICS_DIR = PROJECT_ROOT / "metrics_export"

import os
import re
import unicodedata

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

os.environ["HF_HUB_DISABLE_XET"] = "1"

RANDOM_SEED = 15

VALIDATION_SHARE = 0.20

NEAR_DUPLICATE_SIMILARITY = 0.90

DATASET_SOURCES = {
    "deepset": {
        "repo": "deepset/prompt-injections",
        "revision": "4f61ecb038e9c3fb77e21034b22511b523772cdd",
        "files": {
            "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
            "test": "data/test-00000-of-00001-701d16158af87368.parquet",
        },
    },
    "safeguard": {
        "repo": "xTRam1/safe-guard-prompt-injection",
        "revision": "a3a877d608f37b7d20d9945671902df895ecdb46",
        "files": {
            "train": "data/train-00000-of-00001.parquet",
            "test": "data/test-00000-of-00001.parquet",
        },
    },
}

SAFEGUARD_SAMPLE_SIZES = {"train": 800, "test": 200}

/home/opc/securityhw/project/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_frames = []

for dataset_name, source in DATASET_SOURCES.items():
    for publisher_split, filename in source["files"].items():
        download_path = hf_hub_download(
            source["repo"],
            filename,
            repo_type="dataset",
            revision=source["revision"], token=False,
        )
        frame = pd.read_parquet(download_path)
        frame["source_index"] = np.arange(len(frame))

        if dataset_name == "safeguard":
            sample_indices, _ = train_test_split(
                np.arange(len(frame)),
                train_size=SAFEGUARD_SAMPLE_SIZES[publisher_split],
                stratify=frame["label"],
                random_state=RANDOM_SEED,
            )
            frame = frame.iloc[np.sort(sample_indices)].copy()

        frame["dataset"] = dataset_name
        frame["publisher_split"] = publisher_split
        frame["row_id"] = [
            f"{dataset_name}:{publisher_split}:{index}"
            for index in frame["source_index"]
        ]
        dataset_frames.append(frame)

loaded_prompts = pd.concat(dataset_frames, ignore_index=True)
print("Loaded rows:", len(loaded_prompts))


def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", normalized).strip()


loaded_prompts["normalized"] = loaded_prompts["text"].map(normalize_text)
assert loaded_prompts["label"].isin([0, 1]).all()
assert loaded_prompts["text"].notna().all()
assert loaded_prompts["normalized"].ne("").all()

audit_records = []
for dataset_name, dataset in loaded_prompts.groupby("dataset"):
    audit_records.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "source_columns": 2,
            "negative": int(dataset["label"].eq(0).sum()),
            "positive": int(dataset["label"].eq(1).sum()),
            "positive_pct": 100 * dataset["label"].eq(1).mean(),
            "missing_target": int(dataset["label"].isna().sum()),
            "missing_text": int(dataset["text"].isna().sum()),
            "blank_text": int(dataset["normalized"].eq("").sum()),
            "exact_excess_text": int(dataset.duplicated("text").sum()),
            "exact_excess_text_label": int(dataset.duplicated(["text", "label"]).sum()),
            "normalized_excess": int(dataset.duplicated("normalized").sum()),
        }
    )

audit_results = pd.DataFrame(audit_records)
display(audit_results)
print(
    "Entity and timestamp checks: unavailable; source files contain only text and label."
)

Loaded rows: 1662


,dataset,rows,source_columns,negative,positive,positive_pct,missing_target,missing_text,blank_text,exact_excess_text,exact_excess_text_label,normalized_excess
0,deepset,662,2,399,263,39.728097,0,0,0,0,0,0
1,safeguard,1000,2,695,305,30.500000,0,0,0,3,3,3


Entity and timestamp checks: unavailable; source files contain only text and label.


In [3]:
duplicate_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    dtype=np.float32,
)
duplicate_features = duplicate_vectorizer.fit_transform(loaded_prompts["normalized"])
text_similarities = cosine_similarity(duplicate_features)

# The upper triangle counts each related pair once, without self-pairs.
related_pairs = np.triu(text_similarities >= NEAR_DUPLICATE_SIMILARITY, k=1)
left_indices, right_indices = np.where(related_pairs)
group_parents = np.arange(len(loaded_prompts))


def find_group_root(row_index):
    while group_parents[row_index] != row_index:
        group_parents[row_index] = group_parents[group_parents[row_index]]
        row_index = group_parents[row_index]
    return row_index


def merge_groups(first_index, second_index):
    first_root = find_group_root(first_index)
    second_root = find_group_root(second_index)
    if first_root != second_root:
        group_parents[second_root] = first_root


for first_index, second_index in zip(left_indices, right_indices):
    merge_groups(first_index, second_index)

for matching_indices in loaded_prompts.groupby("normalized").indices.values():
    for row_index in matching_indices[1:]:
        merge_groups(matching_indices[0], row_index)

loaded_prompts["group"] = [
    find_group_root(row_index) for row_index in range(len(loaded_prompts))
]

near_duplicate_records = []
for dataset_name in DATASET_SOURCES:
    both_in_dataset = (
        loaded_prompts["dataset"].iloc[left_indices].to_numpy() == dataset_name
    ) & (loaded_prompts["dataset"].iloc[right_indices].to_numpy() == dataset_name)
    dataset_pairs = list(
        zip(
            left_indices[both_in_dataset],
            right_indices[both_in_dataset],
        )
    )
    affected_rows = set()
    nonexact_pair_count = 0
    for first_index, second_index in dataset_pairs:
        affected_rows.update([first_index, second_index])
        if (
            loaded_prompts["text"].iloc[first_index]
            != loaded_prompts["text"].iloc[second_index]
        ):
            nonexact_pair_count += 1

    near_duplicate_records.append(
        {
            "dataset": dataset_name,
            "near_pairs": len(dataset_pairs),
            "nonexact_near_pairs": nonexact_pair_count,
            "rows_in_near_pairs": len(affected_rows),
        }
    )

near_duplicate_audit = pd.DataFrame(near_duplicate_records)
display(near_duplicate_audit)

,dataset,near_pairs,nonexact_near_pairs,rows_in_near_pairs
0,deepset,28,28,49
1,safeguard,17,14,24


In [4]:
loaded_prompts["split"] = loaded_prompts["publisher_split"]
for dataset_name in DATASET_SOURCES:
    publisher_training = loaded_prompts["dataset"].eq(dataset_name) & loaded_prompts[
        "publisher_split"
    ].eq("train")
    training_indices, validation_indices = train_test_split(
        loaded_prompts.index[publisher_training].to_numpy(),
        test_size=VALIDATION_SHARE,
        stratify=loaded_prompts.loc[publisher_training, "label"],
        random_state=RANDOM_SEED,
    )
    loaded_prompts.loc[training_indices, "split"] = "train"
    loaded_prompts.loc[validation_indices, "split"] = "validation"

# Preserve evaluation rows: test takes priority over validation, then training.
split_priority = {"train": 0, "validation": 1, "test": 2}
loaded_prompts["priority"] = loaded_prompts["split"].map(split_priority)
highest_group_priority = loaded_prompts.groupby("group")["priority"].transform("max")
survives_boundary_check = loaded_prompts["priority"].eq(highest_group_priority)

surviving_prompts = loaded_prompts.loc[survives_boundary_check].sort_values(
    ["dataset", "publisher_split", "source_index"]
)
retained_indices = surviving_prompts.drop_duplicates("normalized").index
removed_prompts = loaded_prompts.loc[
    ~loaded_prompts.index.isin(retained_indices)
].copy()
retained_prompts = (
    loaded_prompts.loc[retained_indices].sort_index().reset_index(drop=True)
)

removal_counts = (
    removed_prompts.groupby(["dataset", "split"]).size().rename("removed").reset_index()
)
display(removal_counts)

training_prompts = retained_prompts[retained_prompts["split"].eq("train")]
leakage_counts = {}
for evaluation_split in ["test", "validation"]:
    evaluation_prompts = retained_prompts[
        retained_prompts["split"].eq(evaluation_split)
    ]
    for field, check_name in [
        ("text", "exact"),
        ("normalized", "normalized"),
        ("group", "near_groups"),
    ]:
        overlapping_rows = training_prompts[field].isin(set(evaluation_prompts[field]))
        leakage_counts[f"train_{check_name}_{evaluation_split}"] = int(
            overlapping_rows.sum()
        )

assert not any(leakage_counts.values())
test_groups = set(retained_prompts.loc[retained_prompts["split"].eq("test"), "group"])
validation_groups = set(
    retained_prompts.loc[retained_prompts["split"].eq("validation"), "group"]
)
assert test_groups.isdisjoint(validation_groups)

split_counts = (
    retained_prompts.groupby(["dataset", "split"])
    .agg(
        rows=("label", "size"),
        positive=("label", "sum"),
    )
    .reset_index()
)
split_counts["negative"] = split_counts["rows"] - split_counts["positive"]
split_counts["positive_pct"] = 100 * split_counts["positive"] / split_counts["rows"]

retained_prompts[["row_id", "dataset", "split", "label"]].to_csv(
    DATA_DIR / "split.csv", index=False
)
display(split_counts)
print("Training overlap counts:", leakage_counts)

,dataset,split,removed
0,deepset,train,11
1,deepset,validation,1
2,safeguard,test,1
3,safeguard,train,9


,dataset,split,rows,positive,negative,positive_pct
0,deepset,test,116,60,56,51.724138
1,deepset,train,425,158,267,37.176471
2,deepset,validation,109,40,69,36.697248
3,safeguard,test,199,63,136,31.658291
4,safeguard,train,631,192,439,30.427892
5,safeguard,validation,160,48,112,30.000000


Training overlap counts: {'train_exact_test': 0, 'train_normalized_test': 0, 'train_near_groups_test': 0, 'train_exact_validation': 0, 'train_normalized_validation': 0, 'train_near_groups_validation': 0}


In [5]:
training_mask = retained_prompts.split.eq("train").to_numpy()
validation_mask = retained_prompts.split.eq("validation").to_numpy()
test_mask = retained_prompts.split.eq("test").to_numpy()
labels = retained_prompts.label.to_numpy()

# Keep model selection and operating-threshold calibration independent.
validation_indices = np.flatnonzero(validation_mask)
strata = (retained_prompts.loc[validation_mask, "dataset"] + ":" +
          retained_prompts.loc[validation_mask, "label"].astype(str))
selection_indices, calibration_indices = train_test_split(
    validation_indices, test_size=0.5, stratify=strata, random_state=15)
calibration_groups = set(retained_prompts.loc[calibration_indices, "group"])
move_to_calibration = [i for i in selection_indices
                      if retained_prompts.loc[i, "group"] in calibration_groups]
selection_indices = np.setdiff1d(selection_indices, move_to_calibration)
calibration_indices = np.union1d(calibration_indices, move_to_calibration)
selection_mask = np.zeros(len(retained_prompts), dtype=bool)
calibration_mask = np.zeros(len(retained_prompts), dtype=bool)
selection_mask[selection_indices] = True
calibration_mask[calibration_indices] = True
assert not (selection_mask & calibration_mask).any()
assert np.array_equal(selection_mask | calibration_mask, validation_mask)
assert set(retained_prompts.loc[selection_mask, "group"]).isdisjoint(
    retained_prompts.loc[calibration_mask, "group"])
roles = retained_prompts.loc[validation_mask, ["row_id", "dataset", "label"]].copy()
roles["role"] = np.where(selection_mask[validation_mask], "selection", "calibration")
roles.to_csv(DATA_DIR / "validation_roles.csv", index=False)
print("Epoch-selection / calibration rows:", selection_mask.sum(), calibration_mask.sum())
display(roles.groupby(["role", "dataset", "label"]).size())


Epoch-selection / calibration rows: 132 137


role         dataset    label
calibration  deepset    0        35
                        1        21
             safeguard  0        57
                        1        24
selection    deepset    0        34
                        1        19
             safeguard  0        55
                        1        24
dtype: int64